# Day 18 · Part 2: Tuning XGBoost with Optuna and MLflow

**Day 18 is about tracking machine-learning experiments with MLflow.** [Part 1](Notebook.ipynb) trained one Logistic Regression on the Iris data and logged it as one run. In this notebook **Optuna** tries 30 XGBoost models for you, MLflow logs every attempt, and the best model is registered. It has six stages:

1. **Create the experiment** that will hold all the runs.
2. **Prepare the data:** split 569 breast-cancer tumours into training, validation and test sets.
3. **Define the objective function:** train one model and return one score.
4. **Run the tuning study:** Optuna calls the objective 30 times and keeps the best.
5. **View the results in the MLflow UI.**
6. **Register the best model**, so it can be loaded by name.

![Day 18 Part 2 roadmap in six stages. Stage 1, sections 1 to 4, MLflow: create the experiment. Stage 2, section 5, scikit-learn: split the data into 398 training, 85 validation and 86 test rows. Stage 3, section 6, Optuna: define the objective function, which returns the validation log loss. Stage 4, sections 7 to 8, Optuna: run the study for 30 trials; the best is trial-23 with validation log loss 0.0208, and the final model scores test accuracy 0.9186. Stage 5, section 9, MLflow: view 1 parent run holding 30 child runs. Stage 6, sections 10 to 11, MLflow: register the model as breast-cancer-xgboost with the alias champion.](images/day18_xgboost_roadmap.svg)

### How to use this notebook
- Run the cells **from top to bottom**. Each step uses the variables made by the step before it (`X_train`, `study`, `model_info`, ...).
- Each section follows the same pattern: **the idea → the code → what the output means**.
- The numbers in the notes come from one full run. Optuna and XGBoost are both seeded, so you should see the same values.
- Re-running the notebook adds a **new** study run and a **new** model version. It never overwrites the old ones.
- The diagrams are in the `images/` folder.

### Key words

| Word | Meaning |
|---|---|
| **Hyperparameter** | A setting you choose **before** training, such as `max_depth`. The model does not learn it. |
| **Tuning** | Trying many hyperparameter values to find the ones that score best. |
| **Objective function** | A function that trains one model with the given hyperparameters and returns one score. |
| **Trial** | One call of the objective function: one set of hyperparameters, one model, one score. |
| **Study** | The whole search: all the trials together. |
| **Parent / child run** | In MLflow, the study is one parent run, and each trial is a child run nested inside it. |
| **Model Registry** | MLflow's catalogue of named models, with numbered versions and aliases such as `@champion`. |

### Contents

| Stage | Section | What you will learn |
|---|---|---|
| **1. Create the experiment** | 1. Installing the Libraries | `%pip` installs into the notebook's own Python |
| | 2. Importing the Libraries | Which library does which job |
| | 3. Starting the MLflow Server | Sharing Part 1's server and SQLite database |
| | 4. Creating a New Experiment | Experiments, and giving one a description |
| **2. Prepare the data** | 5. Loading and Splitting the Breast Cancer Data | Why tuning needs a **validation** set as well as a test set |
| **3. Define the objective function** | 6. Defining the Objective Function | Hyperparameter ranges, nested runs, and why the score is log loss |
| **4. Run the tuning study** | 7. Running the Hyperparameter Tuning Study | Parent runs, `create_study` and `optimize` |
| | 8. Training the Final Model and Testing It | Re-opening a run, the one-time test, logging the model |
| **5. View the results** | 9. Viewing the Results in the MLflow UI | Finding the runs in the UI and with `search_runs`, and charting the search |
| **6. Register the best model** | 10. Registering the Best Model | Registered models, versions and aliases |
| | 11. Loading the Registered Model and Predicting | Loading a model with `models:/<name>@<alias>` |
| | Summary | The six steps on one page |

# Stage 1 · Create the Experiment

Install the tools, start the MLflow server and create the experiment that will hold every run in this notebook. *Sections 1–4.*

## 1. Installing the Libraries

This notebook needs two libraries that Part 1 did not use:

| Library | Job |
|---|---|
| `xgboost` | The model: a classifier made of many small decision trees, each one correcting the trees before it |
| `optuna` | The search: chooses which hyperparameters to try next |

`%pip` installs into the **same Python the notebook kernel uses**. After installing for the first time, **restart the kernel**, then carry on from section 2.

In [ ]:
%pip install xgboost optuna

## 2. Importing the Libraries

Everything is imported once, at the top. Each import has one job:

| Import | Job | Used in |
|---|---|---|
| `os`, `subprocess`, `sys`, `time`, `urllib.request`, `Path` | Start the MLflow server and check that it answers | 3 |
| `matplotlib.pyplot`, `pandas` | Draw the chart, build result tables | 9, 11 |
| `mlflow` | Experiments, runs, logging and the registry | 4, 6–11 |
| `MlflowClient` | Extra actions: save a chart into a run, describe a model version, set an alias | 9, 10 |
| `infer_signature` | Records the model's input columns and output type | 8 |
| `optuna` | The tuning study | 6, 7 |
| `load_breast_cancer`, `train_test_split` | The dataset and the split | 5 |
| `accuracy_score`, `log_loss`, `roc_auc_score`, ... | Score the predictions | 6, 8 |
| `XGBClassifier` | The model being tuned | 6, 8 |

In [ ]:
# --- Standard library: paths, processes and timing (used to start the server) ---
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

# --- Data tables and charts ---
import matplotlib.pyplot as plt
import pandas as pd

# --- MLflow: tracking runs and the model registry ---
import mlflow
from mlflow import MlflowClient
from mlflow.models import infer_signature

# --- Optuna: searches for the best hyperparameters ---
import optuna

# --- scikit-learn: the dataset, the split and the scores ---
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

# --- XGBoost: the model we tune ---
from xgboost import XGBClassifier

## 3. Starting the MLflow Server

This is the same server cell as section 3 of [Part 1](Notebook.ipynb). Both notebooks share **one** server and **one** database (`mlflow.db`), so the runs from Part 1 and Part 2 sit side by side in the UI.

- If the server is already running (for example, Part 1 started it), this cell reuses it.
- If not, it starts the server in the background and waits until it answers.

In [ ]:
# ---------------------------------------------------------------------------
# Work out where "Day 18" is, no matter how the kernel was launched.
# Order: VS Code notebook path -> __file__ -> nearest folder with a notebook
#        -> current working directory.
# ---------------------------------------------------------------------------


def _find_notebook_dir() -> Path:
    for key in ("__vsc_ipynb_file__", "__file__"):
        value = globals().get(key)
        if value:
            p = Path(value).expanduser()
            if p.exists():
                return p.resolve().parent

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        try:
            if any(candidate.glob("*.ipynb")):
                return candidate
        except OSError:
            pass
    return cwd


BASE_DIR = _find_notebook_dir()
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Pin the working directory. From here on, every relative path (ours,
# MLflow's, the server subprocess's) resolves inside Day 18.
os.chdir(BASE_DIR)

DB_PATH = BASE_DIR / "mlflow.db"
ARTIFACT_DIR = BASE_DIR / "mlartifacts"
LOG_PATH = BASE_DIR / "mlflow_server.log"

HOST, PORT = "127.0.0.1", 5000
TRACKING_URI = f"http://{HOST}:{PORT}"

# Stop MLflow from ever creating the default ./mlruns folder in cwd.
os.environ["MLFLOW_TRACKING_URI"] = TRACKING_URI


def server_is_up() -> bool:
    try:
        with urllib.request.urlopen(f"{TRACKING_URI}/health", timeout=1) as r:
            return r.status == 200
    except OSError:
        return False


if server_is_up():
    print("MLflow server is already running at", TRACKING_URI)
else:
    # Give the child a clean env so MLFLOW_TRACKING_URI cannot fight the
    # explicit --backend-store-uri / --artifacts-destination flags.
    server_env = os.environ.copy()
    server_env.pop("MLFLOW_TRACKING_URI", None)

    server_log = open(LOG_PATH, "w")
    server = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "mlflow",
            "server",
            "--backend-store-uri",
            f"sqlite:///{DB_PATH.as_posix()}",
            "--artifacts-destination",
            ARTIFACT_DIR.as_posix(),
            "--host",
            HOST,
            "--port",
            str(PORT),
        ],
        cwd=BASE_DIR,
        env=server_env,
        stdout=server_log,
        stderr=subprocess.STDOUT,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )

    # The first start creates the database tables, so give it up to 60 seconds
    for _ in range(60):
        if server_is_up() or server.poll() is not None:
            break
        time.sleep(1)

    if server_is_up():
        print("MLflow server started at", TRACKING_URI)
    else:
        print("MLflow server did not start - check", LOG_PATH)

print("Base dir :", BASE_DIR)
print("Database :", DB_PATH)
print("Artifacts:", ARTIFACT_DIR)
print("Log      :", LOG_PATH)


MLflow server is already running at http://127.0.0.1:5000
Base dir : C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18
Database : C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18\mlflow.db
Artifacts: C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18\mlartifacts
Log      : C:\Users\laksh\Desktop\IJSE\came-ijse\Day 18\mlflow_server.log


## 4. Creating a New Experiment

An **experiment** is a folder for related runs. Part 1 used `MLflow Quickstart`. This notebook gets its own experiment, so the 30 trials do not mix with the Iris runs.

`mlflow.set_experiment(name)` **creates** the experiment the first time and **reuses** it after that, so the cell is safe to run again. `mlflow.set_experiment_tags()` adds a description and labels, which show on the experiment page.

In [ ]:
# Step 1: Point this notebook at the MLflow server started in section 3
mlflow.set_tracking_uri(TRACKING_URI)

# Step 2: Create the experiment, or reuse it if it already exists.
#         An experiment is a folder that groups related runs together.
EXPERIMENT_NAME = "Breast Cancer Classification XGBoost Tuning"
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

# Step 3: Describe the experiment. These tags show on the experiment page.
#         "mlflow.note.content" is special: the UI shows it as the description.
mlflow.set_experiment_tags(
    {
        "mlflow.note.content": "Tune an XGBoost classifier on the breast cancer "
        "dataset with Optuna, then register the best model.",
        "project": "Day 18 - hyperparameter tuning",
        "model_type": "XGBClassifier",
    }
)

print("MLflow Version   :", mlflow.__version__)
print("Tracking URI     :", mlflow.get_tracking_uri())
print("Experiment name  :", experiment.name)
print("Experiment ID    :", experiment.experiment_id)
print("Artifact location:", experiment.artifact_location)

### 4.1 What the output means

| Line | Meaning |
|---|---|
| Tracking URI `http://127.0.0.1:5000` | Every log call goes to the server from section 3 |
| Experiment ID | MLflow's number for the experiment. It appears in the UI links (`#/experiments/<id>`) |
| Artifact location `mlflow-artifacts:/<id>` | Models and files for this experiment are saved in `mlartifacts/<id>/`, inside the Day 18 folder |

The text in `mlflow.note.content` appears as the **description** at the top of the experiment page.

# Stage 2 · Prepare the Data

Load the breast cancer dataset and split it three ways: one part to train on, one to compare the trials, one for a final test. *Section 5.*

## 5. Loading and Splitting the Breast Cancer Data

The dataset has **569 tumours**, each described by **30 measurements** of the cell nuclei (radius, texture, smoothness, ...). The target is the diagnosis: `0` = malignant, `1` = benign.

Tuning needs **three** sets, not two:

| Set | Share | Used for | Seen by |
|---|---|---|---|
| Training | 70 % | Fitting each trial's model | Every trial |
| Validation | 15 % | Scoring each trial, so Optuna can compare them | Every trial (scoring only) |
| Test | 15 % | One final, honest check of the winner | **Only** the final model, once |

> **Why not tune on the test set?** Optuna keeps whichever trial scores best. If that score came from the test set, the winner would be chosen partly *because* it happens to suit those 86 rows, and its test score would look better than the model really is. A separate validation set keeps the test set untouched until the end.

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Load the breast cancer dataset
# 569 tumours, 30 measurements each. as_frame=True gives a pandas DataFrame,
# so the columns keep their names (MLflow uses them in the model signature).
# ---------------------------------------------------------------------------
data = load_breast_cancer(as_frame=True)

X = data.data  # 30 feature columns (the measurements)
y = data.target  # 0 = malignant, 1 = benign

# ---------------------------------------------------------------------------
# Step 2: Split into 70 % train, 15 % validation and 15 % test
# train_test_split only cuts in two, so we cut twice:
#   cut 1: 70 % train | 30 % temp
#   cut 2: temp in half -> 15 % validation | 15 % test
# random_state=42 -> the same split every time, so results can be repeated.
# stratify        -> each set keeps the same share of malignant and benign.
# ---------------------------------------------------------------------------

# Training 70% (the other 30 % goes into a temporary set)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Validation 15% and Test 15% (half of the 30 % temporary set each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)


# ---------------------------------------------------------------------------
# Step 3: Check the result
# ---------------------------------------------------------------------------
def describe(name, labels):
    """Print how many rows a set has, its share of the data, and its class counts."""
    counts = labels.value_counts().sort_index()
    share = len(labels) / len(y)
    per_class = ", ".join(f"{data.target_names[i]}={n}" for i, n in counts.items())
    print(f"{name:<11}: {len(labels):>3} rows ({share:5.1%}) -> {per_class}")


# print Classes
print("Classes    :", dict(enumerate(data.target_names.tolist())))
print("Features   :", X.shape[1])
print()

# print training / validation / test
describe("Training", y_train)
describe("Validation", y_val)
describe("Test", y_test)

Classes    : {0: 'malignant', 1: 'benign'}
Features   : 30

Training   : 398 rows (69.9%) -> malignant=148, benign=250
Validation :  85 rows (14.9%) -> malignant=32, benign=53
Test       :  86 rows (15.1%) -> malignant=32, benign=54


### 5.1 What the output means

- **398 / 85 / 86 rows** is about 70 / 15 / 15 %. The second cut splits 171 rows, which cannot be halved exactly, so the test set gets one extra row.
- Thanks to `stratify`, each set has almost the same share of malignant tumours: 148 / 398 = 37.2 %, 32 / 85 = 37.6 % and 32 / 86 = 37.2 %.

# Stage 3 · Define the Objective Function

Write the one function Optuna will call again and again: take a trial, train one model, return one score. *Section 6.*

## 6. Defining the Objective Function

**Parameters vs hyperparameters**

| | Parameters | Hyperparameters |
|---|---|---|
| Example in XGBoost | The splits and leaf values inside each tree | `max_depth`, `learning_rate`, `n_estimators` |
| Who sets them | The model learns them in `fit()` | **You** choose them before `fit()` |
| How good values are found | Training | **Tuning**: try many, keep the best |

**The six hyperparameters we tune**

| Hyperparameter | Range | A bigger value means |
|---|---|---|
| `n_estimators` | 50–500 | More trees: more capacity, slower training |
| `max_depth` | 2–8 | Deeper trees: more detail, more risk of memorising the training rows |
| `learning_rate` | 0.01–0.3 (log scale) | Each tree corrects more: fewer trees needed, but less careful steps |
| `subsample` | 0.6–1.0 | Each tree sees more of the rows: less randomness between trees |
| `colsample_bytree` | 0.6–1.0 | Each tree sees more of the 30 columns: less randomness between trees |
| `min_child_weight` | 1–10 | A leaf needs more data: simpler trees |

**What `objective(trial)` does**, once per trial:

![How the Optuna study and the MLflow runs fit together. The Optuna study, using TPESampler with seed 42 and direction minimize, suggests hyperparameters. The objective function runs inside a nested child run: 1, log the 6 hyperparameters; 2, train XGBClassifier on the 398 training rows; 3, score it on the 85 validation rows; 4, log val_log_loss, val_accuracy and val_roc_auc. It returns the validation log loss to Optuna, which picks the next trial, 30 times in all. Each trial adds one child row to the parent run xgboost-optuna-study, for example trial-00 0.0317, trial-01 0.0873 and trial-23 0.0208, the best. Section 8 adds the best parameters, test metrics and the model to the parent run.](images/day18_tuning_loop.svg)

> This cell only **defines** the function. Nothing is trained until section 7 calls `study.optimize(objective, ...)`.

In [ ]:
# How many hyperparameter combinations Optuna will try (1 trial = 1 model)
N_TRIALS = 30

# Hide Optuna's own long log lines; the objective prints a short line instead
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial: optuna.Trial) -> float:
    """Train one XGBoost model with hyperparameters chosen by Optuna and
    return its validation log loss (lower is better)."""

    # -----------------------------------------------------------------------
    # Step 1: Ask Optuna for a value for each hyperparameter.
    #   suggest_int / suggest_float pick a value inside the range we allow.
    #   log=True -> small values are tried as often as large ones
    #               (0.01-0.03 as often as 0.1-0.3)
    # -----------------------------------------------------------------------
    params = {
        # how many trees are built, one after another
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        # how many questions deep each tree may grow
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        # how much each new tree is allowed to correct the ones before it
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        # share of the rows each tree is trained on (adds variety)
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        # share of the 30 columns each tree may use (adds variety)
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        # bigger value -> a leaf needs more data -> simpler trees
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "early_stopping_rounds": 30,  # stop if no improvement for 30 trees
        "random_state": 42,  # the same hyperparameters always give the same model
    }

    # -----------------------------------------------------------------------
    # Step 2: Open a CHILD run inside the parent "study" run (section 7).
    #   nested=True -> the UI shows this run underneath the parent run.
    # -----------------------------------------------------------------------
    with mlflow.start_run(run_name=f"trial-{trial.number:02d}", nested=True):
        # Step 3: Log this trial's hyperparameters -> child run > Parameters
        mlflow.log_params(params)

        # Step 4: Train a model on the TRAINING set with these hyperparameters
        #   random_state=42 -> the same hyperparameters always give the same model
        model = XGBClassifier(**params, eval_metric="logloss")
        model.fit(X_train, y_train)

        # Step 5: Score it on the VALIDATION set
        #   predict_proba(...)[:, 1] -> the chance of class 1 (benign)
        #   predict(...)             -> the class itself (0 or 1)
        val_proba = model.predict_proba(X_val)[:, 1]
        val_pred = model.predict(X_val)
        metrics = {
            "val_log_loss": log_loss(y_val, val_proba),
            "val_accuracy": accuracy_score(y_val, val_pred),
            "val_roc_auc": roc_auc_score(y_val, val_proba),
        }

        # Step 6: Log the scores -> child run > Metrics
        mlflow.log_metrics(metrics)

    print(
        f"Trial {trial.number:02d}: val_log_loss = {metrics['val_log_loss']:.4f}"
        f" | val_accuracy = {metrics['val_accuracy']:.4f}"
    )

    # Step 7: Return the one number Optuna should make as SMALL as possible
    return metrics["val_log_loss"]

### 6.1 Why the score is log loss, not accuracy

Optuna needs **one number** to compare trials, and that number must be able to tell close models apart.

- **Accuracy** on 85 validation rows can only move in steps of 1/85 ≈ 0.0118. In the run behind these notes, 21 of the 30 trials scored exactly **0.9882** (84 of 85 right), so accuracy could not separate them.
- **Log loss** looks at the predicted *probability*, not only the class. A model that says "99 % benign" for a benign tumour scores better than one that says "60 % benign", even though both are right. Confident mistakes are punished hardest.

| Prediction for a tumour that really is benign | Log loss for that row |
|---|---|
| 99 % benign (confident and right) | 0.01 |
| 60 % benign (unsure but right) | 0.51 |
| 10 % benign (confident and wrong) | 2.30 |

The score for a set is the average over its rows. **Lower is better**, so the study uses `direction="minimize"`.

# Stage 4 · Run the Tuning Study

Let Optuna call the objective 30 times inside one parent run, then train the winner and test it once. *Sections 7–8.*

## 7. Running the Hyperparameter Tuning Study

This cell does three things:

1. **Opens a parent run** called `xgboost-optuna-study`. It stays open while the trials run, so every `start_run(nested=True)` inside `objective()` becomes its child. The parent run also gets the **`Training Info`** tag, as in Part 1.
2. **Creates the study.** `direction="minimize"` because lower log loss is better. The **TPE sampler** (Tree-structured Parzen Estimator) is Optuna's default search method:
   - the first 10 trials are **random**, to explore the ranges;
   - after that, it looks at which values gave low scores and tries more values **near those**.
3. **Optimises.** `study.optimize(objective, n_trials=30)` calls `objective` 30 times. Then the best result is logged on the parent run.

> The cell takes under a minute.

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Open the PARENT run. Every trial run inside objective() becomes
#         a child of this run.
# ---------------------------------------------------------------------------
with mlflow.start_run(run_name="xgboost-optuna-study") as parent_run:
    # Step 2: Describe the study -> parent run > Tags / Parameters
    mlflow.set_tag("Training Info", "XGBoost tuned with Optuna on breast cancer data")
    mlflow.log_params(
        {
            "n_trials": N_TRIALS,
            "sampler": "TPESampler",
            "objective_metric": "val_log_loss",
        }
    )

    # Step 3: Create the study (the whole search).
    #   direction="minimize" -> a lower log loss is better
    #   TPESampler(seed=42)  -> the search method; the seed makes the trials repeatable
    study = optuna.create_study(
        study_name="xgboost-breast-cancer",
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )

    # Step 4: Run the search. Optuna calls objective() N_TRIALS times,
    #         and uses the scores so far to choose the next hyperparameters.
    study.optimize(objective, n_trials=N_TRIALS)

    # Step 5: Log the winner on the parent run ("best_" marks it clearly)
    mlflow.log_params(
        {f"best_{name}": value for name, value in study.best_params.items()}
    )
    mlflow.log_metrics(
        {
            "best_val_log_loss": study.best_value,
            "best_trial_number": study.best_trial.number,
        }
    )

# Step 6: Show the result
print()
print("Best trial      :", f"trial-{study.best_trial.number:02d}")
print(f"Best val_log_loss: {study.best_value:.4f}")
print("Best hyperparameters:")
for name, value in study.best_params.items():
    print(
        f"  {name:<17}= {value:.4f}"
        if isinstance(value, float)
        else f"  {name:<17}= {value}"
    )

### 7.1 Reading the trial log

- Each line is one trial, that is, one child run. Optuna counts from **0**, so the runs are `trial-00` to `trial-29`.
- The first 10 trials are random, so their scores jump around: from 0.0317 (trial-00) up to 0.1882 (trial-07).
- From trial 10 on, TPE has learned which region works, and nearly every score stays below 0.05.
- The winner is **trial-23**, with a validation log loss of **0.0208** and these hyperparameters: `n_estimators=122`, `max_depth=5`, `learning_rate=0.2580`, `subsample=0.9173`, `colsample_bytree=0.7015`, `min_child_weight=1`.
- `trial-19` and `trial-28` got all 85 validation rows right (accuracy 1.0000), but they still lost: their probabilities were less confident, so their log loss was higher (see 6.1).

**What was logged where**

| Run | Parameters | Metrics | Tags |
|---|---|---|---|
| Parent `xgboost-optuna-study` | `n_trials`, `sampler`, `objective_metric`, `best_*` | `best_val_log_loss`, `best_trial_number` | `Training Info` |
| Each child `trial-NN` | The 6 hyperparameters | `val_log_loss`, `val_accuracy`, `val_roc_auc` | `mlflow.parentRunId` (added by MLflow) |

## 8. Training the Final Model and Testing It

The trial runs only logged numbers, not models. Now the winner is built for real:

1. **Re-open the parent run** with `mlflow.start_run(run_id=...)`, so the model and test scores land in the same run as the study.
2. **Train** an `XGBClassifier` with `study.best_params` on the same training set. With the same `random_state`, this is the same model as trial-23.
3. **Test it once** on the 86 test rows that no trial has seen.
4. **Log the model** with `mlflow.xgboost.log_model()`. MLflow has one saving format, called a **flavor**, for each library: `mlflow.sklearn`, `mlflow.xgboost`, `mlflow.pytorch`, ...

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Re-open the parent run, so the final model is saved next to the
#         study's results (instead of in a new, separate run).
# ---------------------------------------------------------------------------
with mlflow.start_run(run_id=parent_run.info.run_id):
    # Step 2: Train the final model with the best hyperparameters
    #         (same training set, same random_state as the trials)
    best_model = XGBClassifier(
        **study.best_params, eval_metric="logloss", random_state=42
    )
    best_model.fit(X_train, y_train)

    # Step 3: Test it ONCE on the test set: rows that no trial has seen,
    #         so this is an honest estimate for new patients.
    #   average="macro" -> score each class, then take the plain average
    test_proba = best_model.predict_proba(X_test)[:, 1]
    test_pred = best_model.predict(X_test)
    test_metrics = {
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_precision": precision_score(y_test, test_pred, average="macro"),
        "test_recall": recall_score(y_test, test_pred, average="macro"),
        "test_f1": f1_score(y_test, test_pred, average="macro"),
        "test_roc_auc": roc_auc_score(y_test, test_proba),
        "test_log_loss": log_loss(y_test, test_proba),
    }

    # Step 4: Log the test scores -> parent run > Metrics
    mlflow.log_metrics(test_metrics)

    # Step 5: Record the model's signature (30 input columns -> 1 class label)
    signature = infer_signature(X_train, best_model.predict(X_train))

    # Step 6: Save the model into MLflow (parent run > Artifacts > model)
    #   input_example -> 3 sample rows saved with the model, as a usage example
    model_info = mlflow.xgboost.log_model(
        xgb_model=best_model,
        name="model",
        signature=signature,
        input_example=X_train.head(3),
    )

# Step 7: Show the scores, then a report for each class
for name, value in test_metrics.items():
    print(f"{name:<15}: {value:.4f}")
print("Model URI      :", model_info.model_uri)
print()
print(
    classification_report(y_test, test_pred, target_names=data.target_names, digits=4)
)

### 8.1 What the test results mean

| Score | Validation (trial-23) | Test |
|---|---|---|
| Accuracy | 0.9882 | 0.9186 |
| Log loss | 0.0208 | 0.1494 |

The test score is clearly **lower** than the validation score. This is normal, and it is the reason the test set exists:
- Optuna chose trial-23 **because** it did best on the validation set, so its validation score is a little optimistic.
- The sets are small: one mistake out of 86 test rows costs 1.16 % accuracy. Here 79 of 86 are right and 7 are wrong.

The report for each class shows **where** the mistakes are:
- **malignant**, recall 0.8438: 27 of 32 malignant tumours were found and **5 were missed** (predicted benign).
- **benign**, recall 0.9630: 52 of 54 were right and 2 were flagged as malignant.

For a cancer test, missing a malignant tumour is the more costly mistake, so **malignant recall** is the number to watch.

> **Always compare with a simple baseline.** On this same split, XGBoost with its default settings also scores 0.9186 on the test set, while a Logistic Regression on scaled features scores 0.9884. Tuning finds the best **settings** for XGBoost. It does not prove that XGBoost is the best **model** for this data. Logging a baseline run in the same experiment makes that comparison easy (see "Try it yourself" in the Summary).

# Stage 5 · View the Results in the MLflow UI

Find the study and its 30 trials in the dashboard, then read the same data in code and chart the search. *Section 9.*

## 9. Viewing the Results in the MLflow UI

Open **http://127.0.0.1:5000** (or the link the next cell prints) and follow this path:

```
Experiments ─► Breast Cancer Classification XGBoost Tuning
                 └─► xgboost-optuna-study         (expand it to show the 30 child runs)
                       ├─ Overview  : Parameters (best_*), Metrics (best_val_log_loss, test_*), Tags (Training Info)
                       ├─ Artifacts : model/ (the final model), tuning_history.png (the chart in 9.2)
                       └─ trial-00 … trial-29 : each with its 6 hyperparameters and val_* metrics
```

| To do this | Do this in the UI |
|---|---|
| See the trials | Expand `xgboost-optuna-study` in the runs table |
| Rank the trials | Click the `val_log_loss` column header to sort |
| See which values work best | Tick several trials → **Compare** → the parallel coordinates plot |
| Chart a metric for every run | Switch the runs view from **Table** to **Chart** |
| Find runs by tag | Search for `tags."Training Info" LIKE '%Optuna%'` |

The code below asks MLflow for the same child runs with `mlflow.search_runs()`, so you can check the results without the browser.

In [ ]:
# Step 1: Links to open in the browser
experiment_url = f"{TRACKING_URI}/#/experiments/{experiment.experiment_id}"
parent_url = f"{experiment_url}/runs/{parent_run.info.run_id}"
print("Experiment:", experiment_url)
print("Study run :", parent_url)

# ---------------------------------------------------------------------------
# Step 2: Ask MLflow for this study's child runs: the same rows the UI shows.
#   tags.mlflow.parentRunId -> MLflow adds this tag to every nested run
#   order_by                -> best (lowest) validation log loss first
# ---------------------------------------------------------------------------
trials = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
    order_by=["metrics.val_log_loss ASC"],
)
print("Trial runs found:", len(trials))

# Step 3: Keep a few columns, give them short names, and show the 5 best.
#         MLflow stores parameters as text, so learning_rate is turned back
#         into a number before rounding.
columns = {
    "tags.mlflow.runName": "run",
    "metrics.val_log_loss": "val_log_loss",
    "metrics.val_accuracy": "val_accuracy",
    "params.n_estimators": "n_estimators",
    "params.max_depth": "max_depth",
    "params.learning_rate": "learning_rate",
}
top5 = trials[list(columns)].rename(columns=columns).head(5)
top5["learning_rate"] = top5["learning_rate"].astype(float).round(4)
top5.round(4)

### 9.1 The five best trials

`search_runs()` returns a pandas DataFrame with one row per run. The column names start with the kind of value: `params.…`, `metrics.…` or `tags.…`.

| run | val_log_loss | val_accuracy | n_estimators | max_depth | learning_rate |
|---|---|---|---|---|---|
| trial-23 | 0.0208 | 0.9882 | 122 | 5 | 0.2580 |
| trial-19 | 0.0233 | 1.0000 | 69 | 5 | 0.1869 |
| trial-17 | 0.0236 | 0.9882 | 88 | 5 | 0.2889 |
| trial-24 | 0.0265 | 0.9882 | 152 | 3 | 0.1599 |
| trial-29 | 0.0288 | 0.9882 | 68 | 5 | 0.1946 |

4 of the top 5 use `max_depth=5`, and all of them use a fairly high `learning_rate` (0.16–0.29) with only 68–152 trees (the range allows up to 500). For this small dataset, a few bold trees worked better than many careful ones.

### 9.2 Charting the search

Plot each trial's validation log loss in the order the trials ran, plus a line for the best score found so far. The chart is also saved into the study run, under **Artifacts → tuning_history.png**.

In [ ]:
# Step 1: Put the trials back in the order they ran (trial-00, trial-01, ...)
history = trials.sort_values("tags.mlflow.runName").reset_index(drop=True)
trial_numbers = history.index  # 0, 1, 2, ... (Optuna counts from 0)
scores = history["metrics.val_log_loss"]
best_so_far = scores.cummin()  # the lowest score seen up to each trial

# Step 2: Shade the first 10 trials. TPE picks these at random to explore,
#         and only starts learning from the scores after them.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.axvspan(-0.5, 9.5, color="grey", alpha=0.10)
ax.text(
    4.5,
    scores.max(),
    "first 10 trials:\nrandom exploration",
    ha="center",
    va="top",
    fontsize=9,
    color="dimgrey",
)

# Step 3: Draw every trial's score, plus a line for the best score so far
ax.scatter(
    trial_numbers, scores, color="#2a78d6", label="Validation log loss of each trial"
)
ax.step(
    trial_numbers,
    best_so_far,
    where="post",
    color="#eb6834",
    linewidth=2,
    label="Best validation log loss so far",
)

# Step 4: Ring the best trial and label it and the first trial with real values
best_i = scores.idxmin()
ax.scatter(
    best_i, scores[best_i], s=160, facecolors="none", edgecolors="#eb6834", linewidths=2
)
ax.annotate(
    f"first trial\n{scores[0]:.4f}",
    (0, scores[0]),
    xytext=(10, 10),
    textcoords="offset points",
    fontsize=9,
)
ax.annotate(
    f"best: trial-{best_i:02d}\n{scores[best_i]:.4f}",
    (best_i, scores[best_i]),
    xytext=(-20, 45),
    textcoords="offset points",
    fontsize=9,
    color="#eb6834",
    ha="center",
    arrowprops={"arrowstyle": "-", "color": "#eb6834"},
)

ax.set_title(
    f"Optuna search: validation log loss over {len(history)} trials (lower is better)"
)
ax.set_xlabel("Trial number")
ax.set_ylabel("Validation log loss")
ax.set_xlim(-0.5, len(history) - 0.5)
ax.set_ylim(0, scores.max() * 1.08)
ax.set_xticks(range(0, len(history), 5))
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()

# Step 5: Save the chart into the study run too (parent run > Artifacts)
MlflowClient().log_figure(parent_run.info.run_id, fig, "tuning_history.png")
plt.show()

### 9.3 Reading the chart

- **Grey area (trials 0–9):** TPE's random exploration. The scores are spread widely, from 0.0317 up to 0.1882.
- **Orange line:** steps down each time a new best is found: trial-10 (0.0296), trial-17 (0.0236), trial-19 (0.0233) and trial-23 (0.0208).
- **After trial 10** the points sit lower and closer together. This is TPE focusing on the values that worked.
- **After trial 23** nothing beats the best. More trials might still find a small gain, but each new best is smaller than the one before.

# Stage 6 · Register the Best Model

Give the winning model a name, a version number and an alias, then load it by that name. *Sections 10–11.*

## 10. Registering the Best Model

So far the model lives inside one run, and you would need its ID to find it. The **Model Registry** gives it a permanent name:

| Term | Meaning | Here |
|---|---|---|
| Registered model | A named model in the registry | `breast-cancer-xgboost` |
| Version | Each model registered under that name gets the next number | `1` now; `2`, `3`, ... on re-runs |
| Alias | A label that points at one version, and can be moved | `champion` → the version to use |

**Why aliases?** Code that loads `models:/breast-cancer-xgboost@champion` never has to change. To switch to a better model later, you register it and move the alias. The loading code stays the same.

In [ ]:
# The name of the model in the Model Registry
MODEL_NAME = "breast-cancer-xgboost"

# ---------------------------------------------------------------------------
# Step 1: Register the model logged in section 8.
#   First time -> creates the registered model "breast-cancer-xgboost", version 1.
#   Each re-run of this notebook -> adds version 2, 3, ...
# ---------------------------------------------------------------------------
model_version = mlflow.register_model(model_uri=model_info.model_uri, name=MODEL_NAME)

# MlflowClient gives access to the registry's extra actions
client = MlflowClient()

# Step 2: Describe this version -> Models > breast-cancer-xgboost > Version N
client.update_model_version(
    name=MODEL_NAME,
    version=model_version.version,
    description=(
        f"XGBoost tuned with Optuna ({N_TRIALS} trials, best trial-"
        f"{study.best_trial.number:02d}). Test accuracy {test_metrics['test_accuracy']:.4f}."
    ),
)

# Step 3: Point the alias "champion" at this version.
#   Code that loads "@champion" always gets this version, until the alias is moved.
client.set_registered_model_alias(MODEL_NAME, "champion", model_version.version)

# Step 4: Show what is in the registry now
champion_version = client.get_model_version_by_alias(MODEL_NAME, "champion")
print("Registered model:", MODEL_NAME)
print("Version         :", model_version.version)
print("Alias champion  -> version", champion_version.version)
print("Registry page   :", f"{TRACKING_URI}/#/models/{MODEL_NAME}")

### 10.1 What the output means

- `Successfully registered model` appears only the first time. Later runs print `Created version '2'`, `'3'`, and so on.
- **Version 1** was created, and the alias **champion** points at it.
- In the UI, open the **Models** page (top bar) → `breast-cancer-xgboost`. It lists every version with its alias and description. Each version links back to the run that created it.

## 11. Loading the Registered Model and Predicting

Load the model by **name and alias**, the way another program or a teammate would. `mlflow.pyfunc.load_model()` returns a general model with a `.predict()` method, whatever library trained it.

In [ ]:
# Step 1: Load the model by name and alias: "models:/<name>@<alias>".
#         No run ID needed - the alias always points at the chosen version.
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")

# Step 2: Predict for every patient in the test set (0 = malignant, 1 = benign)
predictions = champion.predict(X_test)

# Step 3: Put the actual and predicted diagnosis side by side
result = pd.DataFrame(
    {
        "actual": data.target_names[y_test],
        "predicted": data.target_names[predictions],
    },
    index=X_test.index,
)
result["correct"] = result["actual"] == result["predicted"]

# Step 4: Count the right answers, then show the first 10 patients
print(f"Correct: {result['correct'].sum()} of {len(result)} test patients")
result.head(10)

### 11.1 What the output means

- **79 of 86** correct is 0.9186, the same as the test accuracy in section 8. So the registered model is exactly the model that was tested.
- The first 10 rows include 4 mistakes (rows 38, 73, 41 and 363): three malignant tumours predicted benign, and one benign tumour predicted malignant. These are 4 of the 7 test mistakes, so the first 10 rows happen to hold more than their share.

# Summary

| Step | What you do | Key code | Where to see it in the UI |
|---|---|---|---|
| 1. Create an experiment | Group the runs | `mlflow.set_experiment(name)` | Experiments list |
| 2. Prepare the data | 70 / 15 / 15 split | `train_test_split(..., stratify=y)`, twice | – |
| 3. Define the objective function | Train one model, return one score | `trial.suggest_*()` + `mlflow.start_run(nested=True)` | The child runs |
| 4. Run the tuning study | Try 30 models, keep the best | `study.optimize(objective, n_trials=30)` | Parent run: `best_*`, `test_*`, model |
| 5. View the results | Compare the trials | `mlflow.search_runs(...)` | Runs table, Compare, Chart view |
| 6. Register the best model | Name, version and alias | `mlflow.register_model(...)`, `set_registered_model_alias(...)` | Models page |

**Remember**
- Tune on the **validation** set. Use the **test** set once, at the end.
- One study = one **parent** run. One trial = one **child** run (`nested=True`).
- Load models by **alias** (`models:/name@champion`), not by run ID.

**Try it yourself**
1. Set `N_TRIALS = 60` and run sections 6–11 again. Does the best log loss improve? A new parent run and model version 2 appear.
2. Add `"reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True)` to the objective and see whether it helps.
3. Log a Logistic Regression run in this experiment as a baseline, and compare it with the tuned XGBoost in the UI.

The MLflow server keeps running in the background after this notebook. Section 9 of [Part 1](Notebook.ipynb) shows how to stop it.